In [ ]:
# File: large-animals.ipynb
# Code: Claude Code and Codex
# Review: Ryoichi Ando (ryoichi.ando@zozo.com)
# License: Apache v2.0

In [ ]:
import os
import zipfile

from frontend import App, get_cache_dir

# create an app
app = App.create("large-animals")

# fetch the volumetric assets on first use and cache them locally. They ship as
# a single archive, so expand it once and reuse it on later runs.
asset_root = os.path.join(get_cache_dir(), "tet-assets")
app.extra.sparse_clone(
    "https://github.com/wiso-enoji/Barrier-Free-Supplementary",
    asset_root,
    ["assets.zip"],
)
asset_dir = os.path.join(asset_root, "assets")
if not os.path.isdir(asset_dir):
    with zipfile.ZipFile(os.path.join(asset_root, "assets.zip")) as archive:
        archive.extractall(asset_root)

# load the stack of soft bodies (1.34M tetrahedra) and the open container
V, F, T = app.mesh.load_tet(os.path.join(asset_dir, "animal_well.1.mesh"))

# The asset reaches y = 20.16. Recentering about the midpoint brings the extent
# to +/-10.08, the smallest coordinate magnitude this geometry can be placed at.
# That magnitude is what float resolution scales with, and every contact gap is
# resolved against it. It has to be done to the mesh itself: an object's
# placement is stored as a separate displacement, so .at() moves the object
# without changing the coordinates the geometry is stored in. This is a pure
# translation, so the dynamics are untouched.
lift = 0.5 * (V[:, 1].min() + V[:, 1].max())
V[:, 1] -= lift
app.asset.add.tet("animals", V, F, T)

V, F = app.mesh.load_tri(os.path.join(asset_dir, "pool.obj"))
V[:, 1] -= lift  # same shift, so the container still sits under the stack
app.asset.add.tri("pool", V, F)

# create a scene
scene = app.scene.create()

# Fixed container the bodies collapse into, marked invisible so its walls do
# not stand between the camera and the stack. That is a drawing flag only, so
# the bodies still collide with it exactly as before.
scene.add("pool").invisible().pin()

# the falling stack. Young's modulus is pre-normalized by density here (the
# elastic energy density is scaled by mass), so divide it through.
density, young_mod = 1e3, 5e5
(
    scene.add("animals")
    .param.set("density", density)
    .set("young-mod", young_mod / density)
    .set("poiss-rat", 0.3)
    .set("model", "snhk")
)

# compile the scene and report stats
scene = scene.build().report()

# preview the initial scene
scene.preview()

In [ ]:
# create a new session with the compiled scene
session = app.session.create(scene)

# set session parameters
(
    session.param.set("auto-save", 10)
    .set("dt", 0.01)
    .set("fps", 100)
    .set("frames", 300)
    .set("gravity", [0, -9.81, 0])
    .set("target-toi", 0.999)
    .set("cg-tol", 0.0001)
    .set("csrmat-max-nnz", 10000000)
)

# build this session
session = session.build()

In [ ]:
# start the simulation (this example takes a long time)
session.start()

In [ ]:
# this example takes a long time...
# in case you shutdown the server (or kernel) and still want to restart
# from where you have (auto) saved, do this. Do not call cells above.

from frontend import App  # noqa

# recover the session from auto-saved state
session = App.recover("large-animals")

# resume if not currently running
if not App.busy():
    session.resume()

# preview the current state
session.preview()

# stream the logs
session.stream()

In [ ]:
session.animate()